# Jupyter Notebook and JSON

This notebook explains what JSON is, how a Jupyter Notebook is encoded as JSON,
and demonstrates (with example code) how to extract structured experimental data from JSON files and from `.ipynb` files.

All code cells are examples only — do not run them automatically here.

## What is JSON?

- **JSON** (JavaScript Object Notation) is a lightweight, text-based, language-independent data interchange format.
- It represents data as nested dictionaries/objects, lists/arrays, strings, numbers, booleans and null.
- JSON is human-readable and easy for programs to parse and generate.

Example JSON (a minimal experiment record):

```json
{
  "experiment_id": "exp-001",
  "date": "2026-02-06",
  "measurements": [
    { "sample": "A", "value": 1.23, "units": "mM" },
    { "sample": "B", "value": 2.34, "units": "mM" }
  ]
}
```

## How a Jupyter Notebook is JSON

- A `.ipynb` file *is* JSON. The top-level JSON object contains metadata and a `cells` array.
- Each cell is represented as a JSON object with `cell_type`, `metadata`, and `source` (an array of strings).
- This means notebooks can be programmatically inspected and parsed like any other JSON file — which enables automatic extraction of structured experimental records.


In [1]:
# Example: read an .ipynb file as JSON and inspect its cells
import json
from pathlib import Path

nb_path = Path("01-JN_and_JSON.ipynb")
with nb_path.open('r', encoding='utf-8') as f:
    nb = json.load(f)

# nb is a dict; get the list of cells
cells = nb.get('cells', [])
print(f"Found {len(cells)} cells")
# Examine the first cell's type and first line of source
if cells:
    first = cells[0]
    print('type:', first.get('cell_type'))
    src = ''.join(first.get('source', []))[:200]
    print('preview:', src)


Found 8 cells
type: markdown
preview: # Jupyter Notebook and JSON

This notebook explains what JSON is, how a Jupyter Notebook is encoded as JSON,
and demonstrates (with example code) how to extract structured experimental data from JSON 


## Extracting structured data from JSON (example)

Below is an example where experimental metadata and measurements are stored as JSON and are converted into a `pandas.DataFrame` for analysis and storage.

In [3]:
# Example: parse experimental JSON into a DataFrame
import json
import pandas as pd

# Example JSON record (could be loaded from a file)
experiment_json = {
    'experiment_id': 'exp-001',
    'date': '2026-02-06',
    'operator': 'Dr. Example',
    'measurements': [
        {'sample': 'A', 'value': 1.23, 'units': 'mM'},
        {'sample': 'B', 'value': 2.34, 'units': 'mM'},
    ]
}

# Normalize the nested 'measurements' list into a table
df = pd.json_normalize(experiment_json, 'measurements', ['experiment_id', 'date', 'operator'])
# df now has columns: sample, value, units, experiment_id, date, operator
print(df.head())
# Optionally save to CSV for downstream tools:
# df.to_csv('exp-001_measurements.csv', index=False)


  sample  value units experiment_id        date     operator
0      A   1.23    mM       exp-001  2026-02-06  Dr. Example
1      B   2.34    mM       exp-001  2026-02-06  Dr. Example


## Using automatic extraction for an Electronic Laboratory Notebook (ELN)

- **Consistency**: Store experimental metadata and results as structured JSON fields (experiment ID, date, operator, parameters, measurements).
- **Automation**: Scripts can scan `.ipynb` files (JSON) to extract experiment sections and measurement tables automatically.
- **Provenance**: Notebook cell metadata (and timestamps in file history / git commits) give provenance for results.
- **Interoperability**: JSON is easily consumed by databases, LIMS, data-analysis pipelines, and web services.

Practical workflow ideas:
- Define a lightweight JSON schema or metadata header cell in each notebook containing experiment-level fields.
- Use a dedicated code cell to append structured measurement results to a JSON file or push to an API at the end of an experiment.
- Create small utilities that parse `.ipynb` files and extract any JSON blobs or tabular outputs into a central ELN database.

| Method | Pros | Cons (Why avoid for aggregation) |
| :--- | :--- | :--- |
| **1. File Export** (Recommended) | Simple, language-agnostic, structured, and fast reading. | Requires an extra line of code in the template. |
| **2. Direct IPYNB Read** (e.g., `nbformat`) | No extra output files needed. | Complex parsing (reading cell outputs as strings), prone to error if output format changes, and requires the notebook to be fully executed before reading. |
| **3. Named Prints/Tags** | Simple to implement initially. | The output is **unstructured text**, making reliable programmatic reading and type conversion (e.g., string to float) difficult and fragile. |


## Next steps and suggested exercises

1. Add a top-level metadata cell to your experiment notebooks with a JSON object named `experiment_metadata`.
2. Practice writing a small script (like the examples above) that walks a folder of `.ipynb` files and extracts `experiment_metadata` plus any tables produced into CSV files.
3. Consider storing extracted JSON in a database or pushing to an ELN/LIMS API.

If you want, I can:
- add a small extractor script file to this repository, or
- run a non-destructive dry-run extractor on the current notebook to show the extracted data preview (requires your permission).